# Sentinel_AI – Advanced Fraud Detection Model Explanation
## Understanding the data, model, evaluation metrics, and business impact

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, precision_recall_curve, roc_curve)
from src.data import generate_fraud_data, preprocess_data
from src.features import engineer_features

%matplotlib inline
os.makedirs("images", exist_ok=True)

## 1. Data Overview & Class Imbalance

Fraud datasets are typically highly imbalanced – legitimate transactions far outnumber fraudulent ones. Our synthetic data reflects this (fraud rate ~1–2%).

In [ ]:
df = generate_fraud_data(10000)
fraud_rate = df['is_fraud'].mean()
print(f"Fraud rate: {fraud_rate:.2%}")
df['is_fraud'].value_counts().plot(kind='bar', title='Class Distribution')
plt.savefig("images/class_distribution.png", bbox_inches='tight')
plt.show()

## 2. Feature Engineering

We create features that capture fraud patterns:
- **`amount_z_score`**: Per‑user standardised transaction amount – unusually large amounts signal risk.
- **`transaction_velocity`** (optional): Number of transactions in last hour – rapid succession fraud.
- **`rolling_fraud_rate`** (optional): Recent fraud proportion per user – temporal pattern.

For this notebook, we enable only `amount_z_score` to match production settings.

In [ ]:
df = engineer_features(df, config={'velocity': False, 'z_score': True, 'rolling_fraud': False})
df[['amount', 'amount_z_score']].head()

## 3. Model Training (Random Forest)

We use `RandomForestClassifier` with `class_weight='balanced'` to handle imbalance. This automatically assigns higher weight to the minority (fraud) class, making the model more sensitive to fraud.

In [ ]:
X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

## 4. Evaluation Metrics Deep Dive

For imbalanced fraud detection, **accuracy is misleading**. We focus on:
- **ROC‑AUC**: Measures the model's ability to rank fraud higher than non‑fraud. Value ranges 0.5 (random) to 1.0 (perfect).
- **Average Precision (AP)**: Also called PR‑AUC – more sensitive to rare class performance. Preferred over ROC‑AUC when fraud rate < 5%.
- **Precision & Recall**: Precision = true frauds / (true frauds + false alarms). Recall = true frauds / (all actual frauds).

In [ ]:
auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.4f}")
print(f"Average Precision: {ap:.4f}")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, label=f'ROC (AUC={auc:.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig("images/roc_curve.png", bbox_inches='tight')
plt.show()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
plt.plot(rec, prec, label=f'PR (AP={ap:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision‑Recall Curve')
plt.legend()
plt.savefig("images/pr_curve.png", bbox_inches='tight')
plt.show()

print("\nClassification Report (with zero_division=0 to avoid warnings):")
print(classification_report(y_test, y_pred, zero_division=0))

## 5. Threshold Selection & Business Impact

The default threshold of 0.5 is rarely optimal for fraud. We choose a threshold that balances **precision** (avoid false alarms) and **recall** (catch fraud).

### Method 1: Target Precision
Set a desired precision (e.g., 80%) and find the threshold that achieves it.

### Method 2: Maximise F1 score
Find threshold that maximises the harmonic mean of precision and recall.

### Business Impact Formula
```
Expected savings = (Total transaction value × Fraud rate × Recall) - (Alerts × Review cost)
```
Example: $100M monthly volume, 1% fraud rate → $1M fraud loss. With recall 0.6 and 5,000 alerts at $5 review, net savings = $600k - $25k = $575k per month.

In [ ]:
def find_optimal_threshold(y_true, y_proba, target_precision=None):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    if target_precision is not None:
        for i, p in enumerate(precisions[:-1]):
            if p >= target_precision:
                return thresholds[i]
    # Else maximise F1
    f1 = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1)
    return thresholds[best_idx]

thresh_f1 = find_optimal_threshold(y_test, y_proba)
thresh_prec80 = find_optimal_threshold(y_test, y_proba, target_precision=0.8)
print(f"Threshold maximizing F1: {thresh_f1:.3f}")
print(f"Threshold for 80% precision: {thresh_prec80:.3f}")

# Apply threshold
y_pred_opt = (y_proba >= thresh_f1).astype(int)
print("\nClassification Report with optimal threshold:")
print(classification_report(y_test, y_pred_opt, zero_division=0))

## 6. How to Use the Model for Predictions

Once trained, the model can score new transactions. Below we demonstrate on a sample.

In [ ]:
# Take first 5 test samples
sample = X_test[:5]
probs = model.predict_proba(sample)[:, 1]
print("Fraud probabilities:", probs)

# Flag using optimal threshold
flags = (probs >= thresh_f1).astype(int)
print("Flag as fraud (1) or not (0):", flags)